# Scaled LLM Q&A Training Pipeline

Train personalized Q&A LLMs for ESP32-S3 inference via llama2.c.

## Training Phases
1. **P1 Stories**: Learn sentence formation from TinyStories
2. **P2 Profile**: Ground in personal facts (prose + synthetic Q&A)
3. **P3 Q&A**: Instruction tuning with full.csv dataset

## Output
- Model: `ajv009_{SIZE}M_{VOCAB}TK.bin` (e.g., `ajv009_4M_1024TK.bin`)
- Tokenizer: `ajv009_{VOCAB}TK.bin`

## Configuration

In [1]:
#=============================================================================
# CONFIGURATION - Modify these to control training
#=============================================================================

# Model Architecture
MODEL_SIZE = "6M"  # Options: "1M", "2M", "3M", "4M", "5M", "6M"
VOCAB_SIZE = 1024  # Options: 512, 768, 1024

# Training Strategy
USE_PRETRAINED = False  # True = fine-tune stories15M, False = train from scratch
TRAINING_PHASES = ["P1_stories", "P2_profile", "P3_qa"]  # Can remove phases to skip

# Hardware (adjust based on GPU)
# V100 (32GB): 64-128, A40 (48GB): 128-256, A100 (40GB): 128-256
BATCH_SIZE = 128
DEVICE = "cuda"

# Training steps per phase (will auto-scale with model size)
BASE_STEPS = {
    "P1_stories": 10000,  # Language learning
    "P2_profile": 5000,   # Profile grounding (no stories now)
    "P3_qa": 6000,        # Q&A memorization (increased from 3000)
}

# Early stopping
EARLY_STOPPING_PATIENCE = 1000  # Steps without improvement
MIN_DELTA = 0.001  # Minimum loss improvement

# Logging
LOG_EVERY = 100  # Steps between log entries
CHECKPOINT_EVERY = 1000  # Steps between checkpoints
PLOT_EVERY = 100  # Steps between plot updates

# Output naming
OUTPUT_PREFIX = f"ajv009_{MODEL_SIZE}_{VOCAB_SIZE}TK"

print(f"Configuration:")
print(f"  Model: {MODEL_SIZE} params, vocab={VOCAB_SIZE}")
print(f"  Pretrained: {USE_PRETRAINED}")
print(f"  Phases: {TRAINING_PHASES}")
print(f"  Output: {OUTPUT_PREFIX}")

Configuration:
  Model: 6M params, vocab=1024
  Pretrained: False
  Phases: ['P1_stories', 'P2_profile', 'P3_qa']
  Output: ajv009_6M_1024TK


In [2]:
# Model size configurations
MODEL_CONFIGS = {
    "1M": {"dim": 128, "hidden_dim": 512, "n_layers": 4, "n_heads": 8, "n_kv_heads": 4},
    "2M": {"dim": 160, "hidden_dim": 640, "n_layers": 5, "n_heads": 8, "n_kv_heads": 4},
    "3M": {"dim": 176, "hidden_dim": 704, "n_layers": 6, "n_heads": 8, "n_kv_heads": 4},
    "4M": {"dim": 192, "hidden_dim": 768, "n_layers": 6, "n_heads": 8, "n_kv_heads": 4},
    "5M": {"dim": 208, "hidden_dim": 832, "n_layers": 7, "n_heads": 8, "n_kv_heads": 4},
    "6M": {"dim": 224, "hidden_dim": 896, "n_layers": 8, "n_heads": 8, "n_kv_heads": 4},
}

# Get current model config
MODEL_CONFIG = MODEL_CONFIGS[MODEL_SIZE]
MODEL_CONFIG["vocab_size"] = VOCAB_SIZE
MODEL_CONFIG["max_seq_len"] = 256
MODEL_CONFIG["dropout"] = 0.1

# Scale training steps based on model size
size_multiplier = {"1M": 0.5, "2M": 0.75, "3M": 1.0, "4M": 1.25, "5M": 1.5, "6M": 2.0}
TRAINING_STEPS = {k: int(v * size_multiplier[MODEL_SIZE]) for k, v in BASE_STEPS.items()}

print(f"\nModel Architecture:")
for k, v in MODEL_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nTraining Steps:")
for phase, steps in TRAINING_STEPS.items():
    print(f"  {phase}: {steps}")


Model Architecture:
  dim: 224
  hidden_dim: 896
  n_layers: 8
  n_heads: 8
  n_kv_heads: 4
  vocab_size: 1024
  max_seq_len: 256
  dropout: 0.1

Training Steps:
  P1_stories: 20000
  P2_profile: 10000
  P3_qa: 12000


## Environment Setup

In [3]:
# Install dependencies
!uv add torch numpy sentencepiece tqdm matplotlib pandas pyyaml datasets openai python-dotenv ipywidgets
print("Dependencies installed!")

Resolved 153 packages in 1ms
Audited 148 packages in 4ms
Dependencies installed!


In [4]:
import torch
print(f'CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

CUDA: True, Device: Tesla V100-PCIE-16GB


In [5]:
# Clone llama2.c
import os
from pathlib import Path

WORK_DIR = Path('/workspace')
LLAMA2C_DIR = WORK_DIR / 'llama2.c'

if not LLAMA2C_DIR.exists():
    !git clone https://github.com/karpathy/llama2.c.git {LLAMA2C_DIR}
else:
    print(f"llama2.c already exists at {LLAMA2C_DIR}")

# Add to path
import sys
sys.path.insert(0, str(LLAMA2C_DIR))

llama2.c already exists at /workspace/llama2.c


In [6]:
# Create directory structure
import os
from pathlib import Path
import json
import csv
from datetime import datetime

WORK_DIR = Path('/workspace')
DATA_DIR = WORK_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
TOKENIZER_DIR = DATA_DIR / 'tokenizer'
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
OUTPUT_DIR = WORK_DIR / 'output'
LOG_DIR = WORK_DIR / 'logs'
PLOT_DIR = WORK_DIR / 'plots'

for d in [RAW_DIR, PROCESSED_DIR, TOKENIZER_DIR, OUTPUT_DIR, LOG_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for phase in TRAINING_PHASES:
    (CHECKPOINT_DIR / phase).mkdir(parents=True, exist_ok=True)

print(f"Directory structure created at {WORK_DIR}")

Directory structure created at /workspace


In [7]:
# Setup logging utilities
import logging
from datetime import datetime

# File logger (for tail -f)
log_file = LOG_DIR / 'training.log'
file_handler = logging.FileHandler(log_file, mode='a')
file_handler.setFormatter(logging.Formatter('[%(asctime)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))

logger = logging.getLogger('training')
logger.setLevel(logging.INFO)
logger.addHandler(file_handler)

# Loss history CSV
loss_csv = LOG_DIR / 'loss_history.csv'
if not loss_csv.exists():
    with open(loss_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['timestamp', 'phase', 'step', 'train_loss', 'val_loss', 'lr'])

def log_step(phase, step, train_loss, val_loss=None, lr=None):
    """Log training step to file and CSV"""
    msg = f"Phase: {phase} | Step: {step} | Loss: {train_loss:.4f}"
    if val_loss:
        msg += f" | Val: {val_loss:.4f}"
    if lr:
        msg += f" | LR: {lr:.2e}"
    logger.info(msg)
    
    with open(loss_csv, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([datetime.now().isoformat(), phase, step, train_loss, val_loss or '', lr or ''])

def log_message(msg):
    """Log general message"""
    logger.info(msg)

print(f"Logging to: {log_file}")
print(f"Loss CSV: {loss_csv}")
print(f"\nTo monitor: tail -f {log_file}")

Logging to: /workspace/logs/training.log
Loss CSV: /workspace/logs/loss_history.csv

To monitor: tail -f /workspace/logs/training.log


In [8]:
# Training state for resume capability
state_file = LOG_DIR / 'training_state.json'

def save_state(phase, step, best_loss, metrics=None):
    """Save training state for resume"""
    state = {
        'phase': phase,
        'step': step,
        'best_loss': best_loss,
        'metrics': metrics or {},
        'timestamp': datetime.now().isoformat(),
        'config': {
            'model_size': MODEL_SIZE,
            'vocab_size': VOCAB_SIZE,
            'use_pretrained': USE_PRETRAINED,
        }
    }
    with open(state_file, 'w') as f:
        json.dump(state, f, indent=2)

def load_state():
    """Load training state for resume"""
    if state_file.exists():
        with open(state_file) as f:
            return json.load(f)
    return None

# Check for existing state
existing_state = load_state()
if existing_state:
    print(f"Found existing training state:")
    print(f"  Phase: {existing_state['phase']}")
    print(f"  Step: {existing_state['step']}")
    print(f"  Best Loss: {existing_state['best_loss']:.4f}")
    print(f"  Timestamp: {existing_state['timestamp']}")
    RESUME = True
else:
    print("No existing state found, starting fresh")
    RESUME = False

Found existing training state:
  Phase: P1_stories
  Step: 6000
  Best Loss: 1.7011
  Timestamp: 2025-11-27T16:53:31.050734


In [9]:
RESUME = False

## Upload Data Files

Upload the following files to `/workspace/data/raw/`:
- `profile.md` - Your _index.md profile
- `full.csv` - Q&A dataset

In [10]:
# Check for required data files
profile_file = RAW_DIR / 'profile.md'
qa_file = RAW_DIR / 'full.csv'

missing = []
if not profile_file.exists():
    missing.append('profile.md')
if not qa_file.exists():
    missing.append('full.csv')

if missing:
    print(f"Missing files in {RAW_DIR}:")
    for f in missing:
        print(f"  - {f}")
    print("\nPlease upload these files before continuing.")
else:
    print(f"All required files present in {RAW_DIR}")
    
    # Load and preview Q&A data
    import pandas as pd
    qa_df = pd.read_csv(qa_file)
    print(f"\nQ&A Dataset: {len(qa_df)} pairs")
    print(qa_df.head())

All required files present in /workspace/data/raw

Q&A Dataset: 2008 pairs
   id                          question                           answer
0   1     What is Alphons email address                  chat@ajv009.com
1   2          What is his phone number                   +91-8237842347
2   3             When was Alphons born                      August 2001
3   4                Where does he live                Maharashtra India
4   5  What languages does Alphons know  English Hindi Marathi Malayalam


## Download TinyStories Dataset

In [11]:
# Download TinyStories dataset
from datasets import load_dataset

stories_file = RAW_DIR / 'tinystories.txt'

if not stories_file.exists():
    print("Downloading TinyStories dataset...")
    ds = load_dataset("roneneldan/TinyStories", split="train")
    
    # Save first 100k stories (enough for our small models)
    with open(stories_file, 'w') as f:
        for i, item in enumerate(ds):
            if i >= 200000:
                break
            f.write(item['text'].strip() + '\n\n')
    
    print(f"Saved 100k stories to {stories_file}")
else:
    print(f"TinyStories already exists: {stories_file}")

# Count stories
with open(stories_file) as f:
    content = f.read()
    story_count = content.count('\n\n')
print(f"Total stories: {story_count}")

TinyStories already exists: /workspace/data/raw/tinystories.txt
Total stories: 1062293


## Tokenizer Training

In [12]:
# Prepare tokenizer corpus (combine all data sources)
import sentencepiece as spm

corpus_file = TOKENIZER_DIR / 'corpus.txt'

print("Preparing tokenizer corpus...")

with open(corpus_file, 'w') as out:
    # Add TinyStories sample
    with open(stories_file) as f:
        stories_text = f.read()
        # Take first 20% for tokenizer training
        out.write(stories_text[:len(stories_text)//5])
    
    # Add profile content if exists
    with open(profile_file) as f:
        profile_text = f.read()
        # Repeat profile 10x to ensure good coverage
        out.write((profile_text + '\n') * 10)
    
    # Add Q&A content if exists
    qa_df = pd.read_csv(qa_file)
    for _, row in qa_df.iterrows():
        out.write(f"<|user|>{row['question']}<|assistant|>{row['answer']}<|end|>\n")
    # Repeat Q&A 5x
    for _ in range(4):
        for _, row in qa_df.iterrows():
            out.write(f"<|user|>{row['question']}<|assistant|>{row['answer']}<|end|>\n")

# Get corpus size
corpus_size = os.path.getsize(corpus_file)
print(f"Corpus file: {corpus_size / 1024 / 1024:.2f} MB")

Preparing tokenizer corpus...
Corpus file: 35.48 MB


In [13]:
# Train SentencePiece tokenizer
tokenizer_prefix = TOKENIZER_DIR / f'ajv009_{VOCAB_SIZE}TK'

if not Path(f"{tokenizer_prefix}.model").exists():
    print(f"Training tokenizer with vocab_size={VOCAB_SIZE}...")
    
    spm.SentencePieceTrainer.train(
        input=str(corpus_file),
        model_prefix=str(tokenizer_prefix),
        vocab_size=VOCAB_SIZE,
        model_type='bpe',
        character_coverage=1.0,
        num_threads=os.cpu_count(),
        user_defined_symbols=['<|user|>', '<|assistant|>', '<|end|>'],
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3,
    )
    
    print(f"Tokenizer saved: {tokenizer_prefix}.model")
else:
    print(f"Tokenizer already exists: {tokenizer_prefix}.model")

Tokenizer already exists: /workspace/data/tokenizer/ajv009_1024TK.model


In [14]:
# Test tokenizer
sp = spm.SentencePieceProcessor()
sp.load(str(tokenizer_prefix) + '.model')

test_texts = [
    "<|user|>What is the name?<|assistant|>Alphons Jaimon<|end|>",
    "Alphons works at Etherwise as a GenAI Engineer.",
    "chat@ajv009.com",
    "Once upon a time, there was a little girl named Lucy.",
]

print(f"Vocab size: {sp.get_piece_size()}")
print("\nTokenization tests:")
for text in test_texts:
    tokens = sp.encode(text)
    decoded = sp.decode(tokens)
    print(f"  Text: {text[:50]}..." if len(text) > 50 else f"  Text: {text}")
    print(f"  Tokens ({len(tokens)}): {tokens[:20]}..." if len(tokens) > 20 else f"  Tokens ({len(tokens)}): {tokens}")
    print()

Vocab size: 1024

Tokenization tests:
  Text: <|user|>What is the name?<|assistant|>Alphons Jaim...
  Tokens (18): [918, 4, 450, 148, 13, 222, 919, 960, 5, 955, 215, 923, 669, 189, 920, 52, 35, 6]

  Text: Alphons works at Etherwise as a GenAI Engineer.
  Tokens (23): [135, 215, 923, 669, 776, 927, 199, 324, 921, 130, 932, 424, 168, 9, 638, 46, 955, 948, 324, 827]...

  Text: chat@ajv009.com
  Tokens (12): [26, 277, 1006, 920, 957, 943, 981, 981, 990, 933, 935, 34]

  Text: Once upon a time, there was a little girl named Lu...
  Tokens (13): [182, 194, 9, 149, 941, 153, 33, 9, 150, 203, 248, 695, 933]



In [15]:
# Export tokenizer to llama2.c format
import struct

tokenizer_bin = TOKENIZER_DIR / f'ajv009_{VOCAB_SIZE}TK.bin'

if not tokenizer_bin.exists():
    print("Exporting tokenizer to llama2.c format...")
    
    # Get all pieces and scores
    vocab = []
    for i in range(sp.get_piece_size()):
        piece = sp.id_to_piece(i)
        score = sp.get_score(i)
        vocab.append((piece, score))
    
    # Find max token length
    max_token_len = max(len(piece.encode('utf-8')) for piece, _ in vocab)
    
    with open(tokenizer_bin, 'wb') as f:
        # Write max_token_length
        f.write(struct.pack('I', max_token_len))
        
        # Write each vocab entry
        for piece, score in vocab:
            piece_bytes = piece.encode('utf-8')
            f.write(struct.pack('f', score))  # score
            f.write(struct.pack('I', len(piece_bytes)))  # length
            f.write(piece_bytes)  # string
    
    print(f"Tokenizer exported: {tokenizer_bin}")
    print(f"Size: {os.path.getsize(tokenizer_bin) / 1024:.1f} KB")
else:
    print(f"Tokenizer binary already exists: {tokenizer_bin}")

Tokenizer binary already exists: /workspace/data/tokenizer/ajv009_1024TK.bin


## Data Preprocessing

In [16]:
# Phase 1: Stories data
p1_file = PROCESSED_DIR / 'p1_stories.txt'

if not p1_file.exists():
    print("Preparing Phase 1 (Stories) data...")
    
    with open(stories_file) as f:
        stories = f.read()
    
    with open(p1_file, 'w') as f:
        f.write(stories)
    
    print(f"Phase 1 data saved: {p1_file}")
else:
    print(f"Phase 1 data exists: {p1_file}")

print(f"Size: {os.path.getsize(p1_file) / 1024 / 1024:.2f} MB")

Phase 1 data exists: /workspace/data/processed/p1_stories.txt
Size: 172.54 MB


In [17]:
# GPT-Powered Profile Extraction
# Load environment variables from .env file

from dotenv import load_dotenv
from openai import OpenAI
import os
import json
import time

# Load .env file (create from .env.sample if missing)
env_file = WORK_DIR / '.env'
if not env_file.exists():
    raise FileNotFoundError(
        f".env file not found at {env_file}\n"
        f"Copy .env.sample to .env and fill in your OpenAI credentials:\n"
        f"  cp .env.sample .env"
    )

load_dotenv(env_file)

# Get OpenAI config from environment (fail if missing)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
OPENAI_MODEL_NAME = os.getenv("OPENAI_MODEL_NAME")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not set in .env file")

print(f"OpenAI Config:")
print(f"  Base URL: {OPENAI_BASE_URL}")
print(f"  Model: {OPENAI_MODEL_NAME}")
print(f"  API Key: {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}")

# Test OpenAI connection
print(f"\nTesting OpenAI connection...")
test_client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
test_response = test_client.chat.completions.create(
    model=OPENAI_MODEL_NAME,
    messages=[{"role": "user", "content": "Say 'OK' if you can read this."}],
    max_tokens=10,
)
test_result = test_response.choices[0].message.content.strip()
print(f"  Test response: {test_result}")
print(f"  Connection OK!")


class ProfileExtractor:
    """GPT-powered profile content extractor and Q&A generator."""

    MAX_RETRIES = 10
    RETRY_DELAY = 3  # seconds

    def __init__(self, api_key: str, base_url: str, model: str):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model

    def _call_gpt(self, system_prompt: str, user_prompt: str, temperature: float = 0.7, **kwargs) -> str:
        """Call GPT with retry logic (max 10 retries, 3s interval). Retries on errors or empty responses.
        
        Args:
            system_prompt: System message for the model
            user_prompt: User message/query
            temperature: Sampling temperature (default 0.7)
            **kwargs: Additional parameters passed to the API (e.g., response_format, max_tokens)
        """
        last_error = None
        for attempt in range(1, self.MAX_RETRIES + 1):
            try:
                # Build API call parameters
                api_params = {
                    "model": self.model,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    "temperature": temperature,
                    "max_tokens": kwargs.pop("max_tokens", 8192),  # Default 8192, can override
                }
                # Add any additional kwargs (e.g., response_format)
                api_params.update(kwargs)
                
                response = self.client.chat.completions.create(**api_params)
                content = response.choices[0].message.content
                
                # Check for empty response
                if not content or not content.strip():
                    raise ValueError("GPT returned empty response")
                
                return content
            except Exception as e:
                last_error = e
                print(f"  Attempt {attempt}/{self.MAX_RETRIES} failed: {e}")
                if attempt < self.MAX_RETRIES:
                    print(f"  Retrying in {self.RETRY_DELAY}s...")
                    time.sleep(self.RETRY_DELAY)
        raise RuntimeError(f"All {self.MAX_RETRIES} attempts failed. Last error: {last_error}")

    def extract_prose(self, profile_content: str) -> list[str]:
        """Extract natural prose paragraphs from profile - generates 50+ diverse paragraphs."""
        system_prompt = """You are a biography writer creating rich detailed prose about a person from their profile data.
Your task is to generate MANY diverse prose paragraphs (50+) that capture every aspect of their identity.

CRITICAL RULES:
1. Write in THIRD PERSON ONLY ("Alphons is..." "He works at..." "His passion is...")
2. Each paragraph: 2-4 sentences flowing naturally like a biography
3. NEVER use markdown bullet points headers or formatting - ONLY plain prose
4. PRESERVE EXACT DETAILS: company names dates percentages email addresses phone numbers URLs
5. Generate DIVERSE paragraphs covering:
   - Professional identity and current role (5+ paragraphs with different angles)
   - Technical skills and expertise areas (5+ paragraphs different skill combos)
   - Work history and career progression (5+ paragraphs per job)
   - Educational background (3+ paragraphs)
   - Projects and achievements (2+ paragraphs per project)
   - Certifications and credentials (2+ paragraphs)
   - Personal interests and hobbies (5+ paragraphs)
   - Personality traits and communication style (3+ paragraphs)
   - Contact information and availability (3+ paragraphs)
   - Fun facts and unique details (3+ paragraphs)

6. VARY the writing style:
   - Some formal ("Alphons Jaimon serves as...")
   - Some casual ("Alphons is the kind of developer who...")
   - Some narrative ("Having started his journey in...")
   - Some descriptive ("With expertise spanning...")

Output: Paragraphs separated by blank lines. Generate AT LEAST 50 paragraphs."""

        user_prompt = f"""Convert this profile into 50+ diverse prose paragraphs covering ALL information:

{profile_content}

Remember:
- Generate AT LEAST 50 paragraphs
- Cover EVERY section multiple times from different angles
- Include ALL contact details skills projects and personal info
- Each paragraph should be 2-4 complete sentences
- Separate paragraphs with blank lines"""

        result = self._call_gpt(system_prompt, user_prompt, temperature=0.7)
        paragraphs = [p.strip() for p in result.split('\n\n') if p.strip() and len(p.strip()) > 30]
        
        if len(paragraphs) < 20:
            print(f"  Warning: Only got {len(paragraphs)} paragraphs making additional call...")
            # Make another call for more content
            more_prompt = f"""Generate 30 MORE unique prose paragraphs about this person focusing on:
- Different phrasings of the same facts
- Career highlights and achievements
- Technical skills in context
- Personal life and interests

{profile_content}

Generate 30 NEW paragraphs (different from typical biography openings)."""
            more_result = self._call_gpt(system_prompt, more_prompt, temperature=0.8)
            more_paragraphs = [p.strip() for p in more_result.split('\n\n') if p.strip() and len(p.strip()) > 30]
            paragraphs.extend(more_paragraphs)
        
        if not paragraphs:
            raise ValueError(f"GPT returned no valid prose paragraphs. Raw response:\n{result[:500]}")
        
        return paragraphs

    def extract_facts(self, profile_content: str) -> dict:
        """Extract structured facts as JSON with enforced JSON response format."""
        system_prompt = """Extract ALL key facts from this profile as comprehensive JSON.

Include these categories with ALL available details:
{
  "personal": {
    "full_name": "...",
    "nickname": "...",
    "email": "...",
    "phone": "...",
    "location": "city state country",
    "birth_date": "...",
    "languages": ["..."],
    "pronouns": "..."
  },
  "current_work": {
    "company": "...",
    "title": "...",
    "responsibilities": ["..."],
    "technologies": ["..."]
  },
  "previous_work": [
    {"company": "...", "title": "...", "duration": "...", "highlights": ["..."]}
  ],
  "education": [
    {"institution": "...", "degree": "...", "field": "...", "year": "...", "grade": "..."}
  ],
  "skills": {
    "programming_languages": ["..."],
    "frameworks": ["..."],
    "tools": ["..."],
    "domains": ["..."]
  },
  "projects": [
    {"name": "...", "description": "...", "technologies": ["..."], "url": "..."}
  ],
  "certifications": ["..."],
  "interests": ["..."],
  "personality": {
    "traits": ["..."],
    "communication_style": "...",
    "work_style": "..."
  },
  "social_links": {"platform": "url"},
  "fun_facts": ["..."]
}

Output ONLY valid JSON."""

        # Use response_format to enforce JSON output
        result = self._call_gpt(
            system_prompt, 
            profile_content, 
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        result = result.strip()
        
        # Fallback cleanup if model still wraps in markdown (shouldn't happen with response_format)
        if result.startswith("```"):
            result = result.split("```")[1]
            if result.startswith("json"):
                result = result[4:]
        
        facts = json.loads(result)
        return facts

    def generate_qa_pairs(self, profile_content: str, facts: dict, count: int = 300) -> list[tuple[str, str]]:
        """Generate diverse Q&A pairs - now generates 300+ pairs with specific categories.
        
        IMPORTANT: Generated Q&A pairs must NOT contain commas or periods to avoid CSV parsing issues.
        """
        
        all_qa_pairs = []
        facts_str = json.dumps(facts, indent=2)
        
        # Category-specific prompts for better coverage
        qa_categories = [
            {
                "name": "Identity & Contact",
                "count": 50,
                "focus": """Focus on:
- Full name and nicknames
- Email address (multiple phrasings: "what is your email" "how to contact" "email address")
- Phone number (multiple phrasings)
- Location (city state country variations)
- Languages spoken
- Pronouns and how to address them"""
            },
            {
                "name": "Current Work",
                "count": 50,
                "focus": """Focus on:
- Current job title and company
- Daily responsibilities
- Technologies used at work
- Team and role
- What they are working on
- Career goals"""
            },
            {
                "name": "Skills & Expertise",
                "count": 50,
                "focus": """Focus on:
- Programming languages (yes/no and experience level)
- Frameworks and libraries
- Tools and platforms
- Domain expertise
- Years of experience
- Skill combinations"""
            },
            {
                "name": "Work History",
                "count": 40,
                "focus": """Focus on:
- Previous jobs and companies
- Career progression
- Notable achievements
- Why they left or joined companies
- Work experience duration"""
            },
            {
                "name": "Education & Certs",
                "count": 30,
                "focus": """Focus on:
- Degrees and institutions
- Field of study
- Graduation year
- Grades and scores
- Certifications
- Online courses"""
            },
            {
                "name": "Projects & Achievements",
                "count": 40,
                "focus": """Focus on:
- Personal projects
- Open source contributions
- Hackathon wins
- Publications
- GitHub repos
- Portfolio work"""
            },
            {
                "name": "Personal & Interests",
                "count": 40,
                "focus": """Focus on:
- Hobbies and interests
- Favorite things (books movies music)
- Personal values
- Life outside work
- Fun facts
- Personality traits"""
            }
        ]
        
        base_system_prompt = """Generate Q&A training data for a personal assistant LLM about this person.

CRITICAL FORMATTING RULES (VERY IMPORTANT):
- NEVER use commas in questions or answers (use spaces or "and" instead)
- NEVER end answers with periods or full stops
- Keep answers as SHORT single phrases or fragments
- Use spaces to separate items instead of commas

EXAMPLES OF CORRECT FORMAT:
- Question: What languages does Alphons know
- Answer: English Hindi Marathi Malayalam
- Question: What is his email
- Answer: chat@ajv009.com
- Question: Does Alphons know Python
- Answer: Yes he is proficient in Python
- Question: Where does he work
- Answer: Etherwise as GenAI Engineer

QUESTION DIVERSITY - use different phrasings:
   - Direct: "What is your email"
   - Casual: "How can I reach you"
   - Third person: "What is Alphons email"
   - Indirect: "Tell me his contact"

ANSWER RULES:
   - CONCISE: Short phrases only (no full sentences if possible)
   - ACCURATE: Use exact details from the profile
   - NO COMMAS: Use spaces or "and" to list items
   - NO PERIODS: Never end with a period

Question types to include:
   - Factual: "What is X"
   - Yes/No: "Does Alphons know Python" -> "Yes"
   - Tell me about: "Tell me about his work"
   - Experience: "How long has he worked at X"

NEVER make up information not in the profile

Output format: question,answer (one per line - the comma is ONLY the delimiter between question and answer)"""

        for category in qa_categories:
            print(f"  Generating {category['count']} Q&A pairs for: {category['name']}...")
            
            user_prompt = f"""Generate exactly {category['count']} Q&A pairs.

{category['focus']}

Profile:
{profile_content[:4000]}

Facts:
{facts_str}

REMEMBER: NO COMMAS in questions or answers! NO PERIODS at end of answers!
Generate {category['count']} Q&A pairs. Format: question,answer (one per line)"""

            try:
                result = self._call_gpt(base_system_prompt, user_prompt, temperature=0.8)
                
                for line in result.strip().split('\n'):
                    line = line.strip()
                    # Skip empty lines headers numbered prefixes
                    if not line or line.startswith('#') or line.startswith('question'):
                        continue
                    # Remove numbering like "1. " or "1) "
                    if line[0].isdigit() and ('. ' in line[:4] or ') ' in line[:4]):
                        line = line.split('. ', 1)[-1] if '. ' in line[:4] else line.split(') ', 1)[-1]
                    
                    if ',' in line:
                        parts = line.split(',', 1)
                        if len(parts) == 2:
                            q, a = parts[0].strip().strip('"\''), parts[1].strip().strip('"\'')
                            # Remove any trailing periods from answers
                            a = a.rstrip('.')
                            if q and a and len(q) > 5 and len(a) > 2:
                                all_qa_pairs.append((q, a))
            except Exception as e:
                print(f"    Warning: Failed to generate {category['name']}: {e}")
                continue
        
        if not all_qa_pairs:
            raise ValueError("GPT returned no valid Q&A pairs across all categories")
        
        # Deduplicate by question (keep first occurrence)
        seen_questions = set()
        unique_pairs = []
        for q, a in all_qa_pairs:
            q_lower = q.lower().strip('?').strip()
            if q_lower not in seen_questions:
                seen_questions.add(q_lower)
                unique_pairs.append((q, a))
        
        print(f"  Total unique Q&A pairs: {len(unique_pairs)}")
        return unique_pairs


# Initialize extractor
extractor = ProfileExtractor(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    model=OPENAI_MODEL_NAME
)
print(f"\nProfileExtractor initialized")

# Verify profile file exists
if not profile_file.exists():
    raise FileNotFoundError(f"Profile file not found: {profile_file}")

# Extract from profile
with open(profile_file) as f:
    profile_content = f.read()

print("\nExtracting prose with GPT (50+ paragraphs)...")
prose = extractor.extract_prose(profile_content)
print(f"  Generated {len(prose)} prose paragraphs")

print("\nExtracting facts with GPT (JSON mode)...")
facts = extractor.extract_facts(profile_content)
print(f"  Extracted {len(facts)} fact categories")

print("\nGenerating synthetic Q&A with GPT (300+ pairs by category)...")
synthetic = extractor.generate_qa_pairs(profile_content, facts, count=300)
print(f"  Generated {len(synthetic)} unique Q&A pairs")

print(f"\n--- Sample Outputs ---")
print(f"Prose paragraph 1: {prose[0][:150]}...")
print(f"Prose paragraph 2: {prose[1][:150]}..." if len(prose) > 1 else "")
print(f"Sample Q&A 1: {synthetic[0]}")
print(f"Sample Q&A 2: {synthetic[1]}" if len(synthetic) > 1 else "")

OpenAI Config:
  Base URL: https://openrouter.ai/api/v1
  Model: x-ai/grok-4.1-fast:free
  API Key: sk-or-v1...38c4

Testing OpenAI connection...
  Test response: OK
  Connection OK!

ProfileExtractor initialized

Extracting prose with GPT (50+ paragraphs)...
  Generated 51 prose paragraphs

Extracting facts with GPT (JSON mode)...
  Extracted 11 fact categories

Generating synthetic Q&A with GPT (300+ pairs by category)...
  Generating 50 Q&A pairs for: Identity & Contact...
  Generating 50 Q&A pairs for: Current Work...
  Generating 50 Q&A pairs for: Skills & Expertise...
  Generating 40 Q&A pairs for: Work History...
  Generating 30 Q&A pairs for: Education & Certs...
  Generating 40 Q&A pairs for: Projects & Achievements...
  Generating 40 Q&A pairs for: Personal & Interests...
  Total unique Q&A pairs: 242
  Generated 242 unique Q&A pairs

--- Sample Outputs ---
Prose paragraph 1: Alphons Jaimon embodies the spirit of "Limitless Ideation," channeling his passion from innovation th

In [18]:
# Phase 2: Profile data ONLY (NO stories - they cause contamination)
# Contains: GPT-extracted prose paragraphs + Synthetic Q&A
import random

p2_file = PROCESSED_DIR / 'p2_mixed.txt'

# Force regeneration - delete if exists
if p2_file.exists():
    p2_file.unlink()
    print(f"Deleted existing {p2_file} for regeneration")

print("Preparing Phase 2 (Profile) data - NO STORIES...")

mixed_data = []

# Add GPT-generated prose paragraphs (70%)
prose_repeat = 100  # Repeat prose to get enough data
for _ in range(prose_repeat):
    for p in prose:
        mixed_data.append(p)
print(f"  Added {len(prose) * prose_repeat} prose paragraphs (GPT-generated)")

# Add GPT-generated synthetic Q&A (30%)
qa_repeat = max(1, len(prose) * prose_repeat * 30 // 70 // len(synthetic))
for _ in range(qa_repeat):
    for q, a in synthetic:
        mixed_data.append(f"<|user|>{q}<|assistant|>{a}<|end|>")
print(f"  Added {len(synthetic) * qa_repeat} synthetic Q&A pairs (GPT-generated)")

# Shuffle
random.seed(42)
random.shuffle(mixed_data)

with open(p2_file, 'w') as f:
    f.write('\n\n'.join(mixed_data))

print(f"\nPhase 2 data saved: {p2_file}")
print(f"  Total samples: {len(mixed_data)}")
print(f"  NO STORIES included (removed to prevent contamination)")

print(f"Size: {os.path.getsize(p2_file) / 1024 / 1024:.2f} MB")

Deleted existing /workspace/data/processed/p2_mixed.txt for regeneration
Preparing Phase 2 (Profile) data - NO STORIES...
  Added 5100 prose paragraphs (GPT-generated)
  Added 2178 synthetic Q&A pairs (GPT-generated)

Phase 2 data saved: /workspace/data/processed/p2_mixed.txt
  Total samples: 7278
  NO STORIES included (removed to prevent contamination)
Size: 1.45 MB


In [19]:
# Phase 3: Q&A instruction tuning (heavy memorization)
p3_file = PROCESSED_DIR / 'p3_qa.txt'

# Force regeneration - delete if exists  
if p3_file.exists():
    p3_file.unlink()
    print(f"Deleted existing {p3_file} for regeneration")

if qa_file.exists():
    print("Preparing Phase 3 (Q&A) data with heavy repetition...")
    
    qa_df = pd.read_csv(qa_file)
    
    # Format as instruction tuning with special tokens
    qa_samples = []
    for _, row in qa_df.iterrows():
        qa_samples.append(f"<|user|>{row['question']}<|assistant|>{row['answer']}<|end|>")
    
    # Heavy repetition for memorization (50x instead of 20x)
    QA_REPETITION = 50
    repeated = qa_samples * QA_REPETITION
    
    # Shuffle
    random.seed(42)
    random.shuffle(repeated)
    
    with open(p3_file, 'w') as f:
        f.write('\n'.join(repeated))
    
    print(f"Phase 3 data saved: {p3_file}")
    print(f"  Original Q&A pairs: {len(qa_samples)}")
    print(f"  Repetition: {QA_REPETITION}x")
    print(f"  After repetition: {len(repeated)} samples")
    print(f"  Special tokens: <|user|>, <|assistant|>, <|end|>")
else:
    print("Skipping Phase 3 - full.csv not found")

if p3_file.exists():
    print(f"Size: {os.path.getsize(p3_file) / 1024 / 1024:.2f} MB")

Deleted existing /workspace/data/processed/p3_qa.txt for regeneration
Preparing Phase 3 (Q&A) data with heavy repetition...
Phase 3 data saved: /workspace/data/processed/p3_qa.txt
  Original Q&A pairs: 2008
  Repetition: 50x
  After repetition: 100400 samples
  Special tokens: <|user|>, <|assistant|>, <|end|>
Size: 7.80 MB


In [20]:
# Tokenize all data files
import numpy as np

def tokenize_file(input_file, output_file, tokenizer, force=False):
    """Tokenize text file to binary token file"""
    if output_file.exists() and not force:
        tokens = np.fromfile(output_file, dtype=np.uint16)
        print(f"  Loaded existing: {len(tokens):,} tokens")
        return len(tokens)
    
    # Delete if exists (force regeneration)
    if output_file.exists():
        output_file.unlink()
        print(f"  Deleted existing {output_file.name}")
    
    with open(input_file) as f:
        text = f.read()
    
    tokens = np.array(tokenizer.encode(text), dtype=np.uint16)
    
    with open(output_file, 'wb') as f:
        f.write(tokens.tobytes())
    
    print(f"  Tokenized: {len(tokens):,} tokens")
    return len(tokens)

# Tokenize each phase - force regeneration for P2 and P3 (modified data)
phase_files = {
    'P1_stories': (p1_file, PROCESSED_DIR / 'p1_stories.bin', False),  # Keep existing
    'P2_profile': (p2_file, PROCESSED_DIR / 'p2_mixed.bin', True),     # Force regen (no stories)
    'P3_qa': (p3_file, PROCESSED_DIR / 'p3_qa.bin', True),             # Force regen (50x rep)
}

token_counts = {}
for phase, (txt_file, bin_file, force) in phase_files.items():
    if txt_file.exists():
        print(f"{phase}:")
        token_counts[phase] = tokenize_file(txt_file, bin_file, sp, force=force)

print(f"\nTotal token counts:")
for phase, count in token_counts.items():
    print(f"  {phase}: {count:,}")

P1_stories:
  Loaded existing: 55,067,627 tokens
P2_profile:
  Deleted existing p2_mixed.bin
  Tokenized: 711,904 tokens
P3_qa:
  Deleted existing p3_qa.bin
  Tokenized: 2,723,800 tokens

Total token counts:
  P1_stories: 55,067,627
  P2_profile: 711,904
  P3_qa: 2,723,800


## Model Definition

In [21]:
# Import model from llama2.c
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Import model architecture
sys.path.insert(0, str(LLAMA2C_DIR))
from model import Transformer, ModelArgs

# Create model
model_args = ModelArgs(
    dim=MODEL_CONFIG['dim'],
    n_layers=MODEL_CONFIG['n_layers'],
    n_heads=MODEL_CONFIG['n_heads'],
    n_kv_heads=MODEL_CONFIG['n_kv_heads'],
    vocab_size=MODEL_CONFIG['vocab_size'],
    max_seq_len=MODEL_CONFIG['max_seq_len'],
    dropout=MODEL_CONFIG['dropout'],
)

model = Transformer(model_args)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model created:")
print(f"  Parameters: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"  Expected fp32 size: {n_params * 4 / 1024 / 1024:.2f} MB")
print(f"  Expected int8 size: {n_params / 1024 / 1024:.2f} MB")

Model created:
  Parameters: 5,566,176 (5.57M)
  Expected fp32 size: 21.23 MB
  Expected int8 size: 5.31 MB


In [22]:
# Dataset class
class TokenDataset(Dataset):
    def __init__(self, bin_path, seq_len=256):
        self.seq_len = seq_len
        self.tokens = np.fromfile(bin_path, dtype=np.uint16).astype(np.int32)
    
    def __len__(self):
        return max(1, len(self.tokens) // self.seq_len)
    
    def __getitem__(self, idx):
        start = idx * self.seq_len
        chunk = self.tokens[start:start + self.seq_len + 1]
        if len(chunk) < self.seq_len + 1:
            chunk = np.pad(chunk, (0, self.seq_len + 1 - len(chunk)))
        x = torch.from_numpy(chunk[:-1].astype(np.int64))
        y = torch.from_numpy(chunk[1:].astype(np.int64))
        return x, y

print("Dataset class defined")

Dataset class defined


## Training Loop

In [23]:
# Training utilities
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

def update_loss_plot(phase, losses, val_losses=None):
    """Update loss curve plot"""
    plt.figure(figsize=(10, 6))
    plt.plot(losses, label='Train Loss', alpha=0.7)
    if val_losses:
        val_steps = list(range(0, len(losses), LOG_EVERY))
        plt.plot(val_steps[:len(val_losses)], val_losses, 'r-', label='Val Loss', alpha=0.7)
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title(f'{phase} Training Progress')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(PLOT_DIR / 'loss_curve.png', dpi=100, bbox_inches='tight')
    plt.close()

def train_phase(phase, data_path, model, tokenizer, config):
    """Train one phase"""
    log_message(f"Phase: {phase} | STARTING")
    
    # Check for resume
    resume_step = 0
    best_loss = float('inf')
    
    existing_state = load_state()
    if existing_state and existing_state['phase'] == phase:
        resume_step = existing_state['step']
        best_loss = existing_state['best_loss']
        
        # Load checkpoint
        ckpt_path = CHECKPOINT_DIR / phase / f'ckpt_{resume_step}.pt'
        if ckpt_path.exists():
            checkpoint = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            model.load_state_dict(checkpoint['model'])
            log_message(f"Phase: {phase} | RESUMED from step {resume_step}")
    
    # Create dataset and dataloader
    dataset = TokenDataset(data_path, seq_len=MODEL_CONFIG['max_seq_len'])
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    
    # Optimizer
    optimizer = model.configure_optimizers(
        weight_decay=0.1,
        learning_rate=config['lr'],
        betas=(0.9, 0.95),
        device_type=DEVICE
    )
    
    # Training
    model.train()
    losses = []
    val_losses = []
    patience_counter = 0
    total_steps = config['steps']
    
    pbar = tqdm(total=total_steps - resume_step, desc=f'{phase}', initial=0)
    step = resume_step
    start_time = time.time()
    
    while step < total_steps:
        for x, y in dataloader:
            if step >= total_steps:
                break
            
            x, y = x.to(DEVICE), y.to(DEVICE)
            
            # Forward pass
            logits = model(x, y)
            loss = model.last_loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            losses.append(loss.item())
            step += 1
            pbar.update(1)
            
            # Logging
            if step % LOG_EVERY == 0:
                avg_loss = np.mean(losses[-LOG_EVERY:])
                log_step(phase, step, avg_loss, lr=config['lr'])
                pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
            
            # Checkpoint
            if step % CHECKPOINT_EVERY == 0:
                ckpt_path = CHECKPOINT_DIR / phase / f'ckpt_{step}.pt'
                torch.save({
                    'model': model.state_dict(),
                    'step': step,
                    'loss': np.mean(losses[-CHECKPOINT_EVERY:]),
                }, ckpt_path)
                log_message(f"Phase: {phase} | CHECKPOINT saved: {ckpt_path.name}")
                
                # Check for best model
                current_loss = np.mean(losses[-CHECKPOINT_EVERY:])
                if current_loss < best_loss - MIN_DELTA:
                    best_loss = current_loss
                    patience_counter = 0
                    torch.save({
                        'model': model.state_dict(),
                        'step': step,
                        'loss': best_loss,
                    }, CHECKPOINT_DIR / phase / 'best.pt')
                    log_message(f"Phase: {phase} | BEST model saved (loss: {best_loss:.4f})")
                else:
                    patience_counter += CHECKPOINT_EVERY
                
                # Save state for resume
                save_state(phase, step, best_loss)
                
                # Early stopping
                if patience_counter >= EARLY_STOPPING_PATIENCE:
                    log_message(f"Phase: {phase} | EARLY STOPPING at step {step}")
                    break
            
            # Plot update
            if step % PLOT_EVERY == 0:
                update_loss_plot(phase, losses)
    
    pbar.close()
    elapsed = time.time() - start_time
    
    log_message(f"Phase: {phase} | COMPLETE | Final loss: {np.mean(losses[-100:]):.4f} | Time: {elapsed/60:.1f} min")
    
    # Load best model for next phase
    best_ckpt = CHECKPOINT_DIR / phase / 'best.pt'
    if best_ckpt.exists():
        checkpoint = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model'])
        log_message(f"Phase: {phase} | Loaded best model (loss: {checkpoint['loss']:.4f})")
    
    return model, best_loss

print("Training utilities defined")

Training utilities defined


In [24]:
# Phase configurations
PHASE_CONFIGS = {
    'P1_stories': {
        'data': PROCESSED_DIR / 'p1_stories.bin',
        'steps': TRAINING_STEPS['P1_stories'],
        'lr': 3e-4,
    },
    'P2_profile': {
        'data': PROCESSED_DIR / 'p2_mixed.bin',
        'steps': TRAINING_STEPS['P2_profile'],
        'lr': 1e-4,  # Lower LR for fine-tuning
    },
    'P3_qa': {
        'data': PROCESSED_DIR / 'p3_qa.bin',
        'steps': TRAINING_STEPS['P3_qa'],
        'lr': 5e-5,  # Very low LR for memorization
    },
}

print("Phase configurations:")
for phase, config in PHASE_CONFIGS.items():
    if phase in TRAINING_PHASES:
        print(f"  {phase}: {config['steps']} steps, LR={config['lr']}")

Phase configurations:
  P1_stories: 20000 steps, LR=0.0003
  P2_profile: 10000 steps, LR=0.0001
  P3_qa: 12000 steps, LR=5e-05


In [25]:
# Run training
print("="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Model: {MODEL_SIZE}, Vocab: {VOCAB_SIZE}")
print(f"Phases: {TRAINING_PHASES}")
print(f"\nMonitor with: tail -f {LOG_DIR}/training.log")
print("="*60)

final_losses = {}

for phase in TRAINING_PHASES:
    config = PHASE_CONFIGS[phase]
    
    # Check if data exists
    if not config['data'].exists():
        print(f"\nSkipping {phase} - data file not found: {config['data']}")
        continue
    
    print(f"\n{'='*60}")
    print(f"PHASE: {phase}")
    print(f"{'='*60}")
    
    model, best_loss = train_phase(phase, config['data'], model, sp, config)
    final_losses[phase] = best_loss

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
for phase, loss in final_losses.items():
    print(f"  {phase}: {loss:.4f}")

STARTING TRAINING
Model: 6M, Vocab: 1024
Phases: ['P1_stories', 'P2_profile', 'P3_qa']

Monitor with: tail -f /workspace/logs/training.log

PHASE: P1_stories
num decayed parameter tensors: 57, with 5,562,368 parameters
num non-decayed parameter tensors: 17, with 3,808 parameters
using fused AdamW: True


P1_stories: 100%|██████████| 14000/14000 [46:11<00:00,  5.05it/s, loss=1.5469] 



PHASE: P2_profile
num decayed parameter tensors: 57, with 5,562,368 parameters
num non-decayed parameter tensors: 17, with 3,808 parameters
using fused AdamW: True


P2_profile: 100%|██████████| 10000/10000 [36:35<00:00,  4.56it/s, loss=0.0126] 



PHASE: P3_qa
num decayed parameter tensors: 57, with 5,562,368 parameters
num non-decayed parameter tensors: 17, with 3,808 parameters
using fused AdamW: True


P3_qa: 100%|██████████| 12000/12000 [40:36<00:00,  4.92it/s, loss=0.2334]


TRAINING COMPLETE
  P1_stories: 1.5507
  P2_profile: 0.0133
  P3_qa: 0.2378


## Export Model

In [26]:
# Export model to llama2.c format
sys.path.insert(0, str(LLAMA2C_DIR))
from export import model_export

# Float32 export
fp32_path = OUTPUT_DIR / f'{OUTPUT_PREFIX}.bin'
print(f"Exporting float32 model to {fp32_path}...")
model_export(model, str(fp32_path), version=0)
print(f"Float32 size: {os.path.getsize(fp32_path) / 1024 / 1024:.2f} MB")

# Int8 quantized export
int8_path = OUTPUT_DIR / f'{OUTPUT_PREFIX}_q80.bin'
print(f"\nExporting int8 model to {int8_path}...")
model_export(model, str(int8_path), version=2)
print(f"Int8 size: {os.path.getsize(int8_path) / 1024 / 1024:.2f} MB")

Exporting float32 model to /workspace/output/ajv009_6M_1024TK.bin...
wrote /workspace/output/ajv009_6M_1024TK.bin
Float32 size: 21.26 MB

Exporting int8 model to /workspace/output/ajv009_6M_1024TK_q80.bin...
BACKOFF: reducing group size to 32 to fit hidden_dim
1/57 quantized (1024, 224) to Q8_0 with max error 0.0012849494814872742
2/57 quantized (224, 224) to Q8_0 with max error 0.0010296106338500977
3/57 quantized (224, 224) to Q8_0 with max error 0.0017006397247314453
4/57 quantized (224, 224) to Q8_0 with max error 0.0012102052569389343
5/57 quantized (224, 224) to Q8_0 with max error 0.0012421198189258575
6/57 quantized (224, 224) to Q8_0 with max error 0.001075994223356247
7/57 quantized (224, 224) to Q8_0 with max error 0.0010541453957557678
8/57 quantized (224, 224) to Q8_0 with max error 0.000823628157377243
9/57 quantized (224, 224) to Q8_0 with max error 0.0008649528026580811
10/57 quantized (112, 224) to Q8_0 with max error 0.0014349371194839478
11/57 quantized (112, 224) to

In [27]:
# Copy tokenizer to output
import shutil

tokenizer_output = OUTPUT_DIR / f'ajv009_{VOCAB_SIZE}TK.bin'
shutil.copy(tokenizer_bin, tokenizer_output)

print("\nFinal output files:")
for f in OUTPUT_DIR.glob('*'):
    size = os.path.getsize(f)
    if size > 1024 * 1024:
        print(f"  {f.name}: {size / 1024 / 1024:.2f} MB")
    else:
        print(f"  {f.name}: {size / 1024:.1f} KB")


Final output files:
  ajv009_1M_512TK.bin: 4.02 MB
  ajv009_1M_512TK_q80.bin: 1.07 MB
  ajv009_512TK.bin: 6.0 KB
  ajv009_3M_1024TK.bin: 9.03 MB
  ajv009_3M_1024TK_q80.bin: 2.82 MB
  ajv009_1024TK.bin: 12.9 KB
  ajv009_6M_1024TK.bin: 21.26 MB
  ajv009_6M_1024TK_q80.bin: 5.98 MB


## Validation

In [28]:
# Quick inference test
import torch.nn.functional as F

def generate(model, tokenizer, prompt, max_tokens=50, temperature=0.0):
    """Generate text from prompt"""
    model.eval()
    
    tokens = tokenizer.encode(prompt)
    tokens = torch.tensor([tokens], dtype=torch.long, device=DEVICE)
    
    with torch.no_grad():
        for _ in range(max_tokens):
            if tokens.size(1) >= MODEL_CONFIG['max_seq_len']:
                break
            
            logits = model(tokens)
            logits = logits[:, -1, :]
            
            if temperature == 0:
                next_token = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                probs = F.softmax(logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            
            tokens = torch.cat([tokens, next_token], dim=1)
            
            # Check for end token
            if tokenizer.decode([next_token.item()]) == '<|end|>':
                break
    
    return tokenizer.decode(tokens[0].tolist())

# Test questions
TEST_QUESTIONS = [
    "What is the name?",
    "What is the email?",
    "Where does Alphons work?",
    "Does Alphons know Python?",
    "Tell me about Alphons.",
    "What is his current job?",
]

print("\nInference Test:")
print("="*60)

for q in TEST_QUESTIONS:
    prompt = f"<|user|>{q}<|assistant|>"
    response = generate(model, sp, prompt, max_tokens=50, temperature=0.0)
    
    # Extract answer
    if '<|assistant|>' in response:
        answer = response.split('<|assistant|>')[-1]
        if '<|end|>' in answer:
            answer = answer.split('<|end|>')[0]
    else:
        answer = response
    
    print(f"Q: {q}")
    print(f"A: {answer.strip()}")
    print()


Inference Test:
Q: What is the name?
A: Alphons Jaimon

Q: What is the email?
A: chat@ajv009.com

Q: Where does Alphons work?
A: Etherwise

Q: Does Alphons know Python?
A: Yes for AI ML work

Q: Tell me about Alphons.
A: More about consciousness transfer and upload

Q: What is his current job?
A: GenAI Engineer



In [29]:
# Build and test CLI inference
print("Building CLI inference tools...")
!cd {LLAMA2C_DIR} && make run && make runq

print("\nTesting float32 model:")
!{LLAMA2C_DIR}/run {fp32_path} -z {tokenizer_output} -i '<|user|>What is the name?<|assistant|>' -n 50 -t 0.0

print("\nTesting int8 model:")
!{LLAMA2C_DIR}/runq {int8_path} -z {tokenizer_output} -i '<|user|>What is the name?<|assistant|>' -n 50 -t 0.0

Building CLI inference tools...
gcc -O3 -o run run.c -lm
gcc -O3 -o runq runq.c -lm
make: 'runq' is up to date.

Testing float32 model:

Testing int8 model:


## Summary

Training complete! Your files are in `/workspace/output/`:

- `ajv009_{SIZE}M_{VOCAB}TK.bin` - Float32 model
- `ajv009_{SIZE}M_{VOCAB}TK_q80.bin` - Int8 quantized model (for ESP32)
- `ajv009_{VOCAB}TK.bin` - Tokenizer

### Next Steps

1. Download the int8 model and tokenizer
2. Copy to SD card: `/models/ajv009_{SIZE}M_{VOCAB}TK_q80.bin`
3. Update ESP32 sketch to use new model path

In [30]:
# Final summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model: {OUTPUT_PREFIX}")
print(f"Architecture: dim={MODEL_CONFIG['dim']}, layers={MODEL_CONFIG['n_layers']}, vocab={VOCAB_SIZE}")
print(f"Parameters: {n_params:,} ({n_params/1e6:.2f}M)")
print(f"\nOutput files:")
for f in sorted(OUTPUT_DIR.glob('*')):
    size = os.path.getsize(f)
    print(f"  {f.name}: {size / 1024 / 1024:.2f} MB" if size > 1024*1024 else f"  {f.name}: {size / 1024:.1f} KB")
print(f"\nLogs: {LOG_DIR}")
print(f"Plots: {PLOT_DIR}")
print(f"Checkpoints: {CHECKPOINT_DIR}")


TRAINING SUMMARY
Model: ajv009_6M_1024TK
Architecture: dim=224, layers=8, vocab=1024
Parameters: 5,566,176 (5.57M)

Output files:
  ajv009_1024TK.bin: 12.9 KB
  ajv009_1M_512TK.bin: 4.02 MB
  ajv009_1M_512TK_q80.bin: 1.07 MB
  ajv009_3M_1024TK.bin: 9.03 MB
  ajv009_3M_1024TK_q80.bin: 2.82 MB
  ajv009_512TK.bin: 6.0 KB
  ajv009_6M_1024TK.bin: 21.26 MB
  ajv009_6M_1024TK_q80.bin: 5.98 MB

Logs: /workspace/logs
Plots: /workspace/plots
Checkpoints: /workspace/checkpoints
